<a href="https://colab.research.google.com/github/houstongolden/bigbounce/blob/main/research/notebooks/bigbounce_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BigBounce GPU Research Notebook

**Purpose:** Run GPU-intensive models (Polymathic AI Walrus/AION) and large-scale astronomical ML on Google Colab Pro.

**Setup:** Runtime → Change runtime type → GPU (A100 recommended)

**Secrets:** Add these in the Colab sidebar (key icon):
- `HUGGINGFACE_TOKEN` — for gated model access
- `ANTHROPIC_API_KEY` — for Claude reasoning calls
- `GOOGLE_AI_API_KEY` — for Gemini multimodal
- `DEEPSEEK_API_KEY` — for DeepSeek R1 verification

**Repo:** [Hubify-Projects/bigbounce](https://github.com/Hubify-Projects/bigbounce)

In [58]:
# ── 0. Install dependencies ─────────────────────────────────────────
!pip install -q transformers torch datasets astroML astroquery \
    anthropic openai google-generativeai requests python-dotenv pandas numpy scipy matplotlib

In [59]:
# ── 1. Load API keys from Colab Secrets ─────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ['HUGGINGFACE_TOKEN'] = userdata.get('HUGGINGFACE_TOKEN')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GOOGLE_AI_API_KEY'] = userdata.get('GOOGLE_AI_API_KEY')
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
    print('Loaded keys from Colab Secrets')
except Exception:
    print('Not in Colab or secrets not set — using environment variables')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected. Change runtime to GPU.')

Loaded keys from Colab Secrets
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)


In [60]:
# ── 2. Clone BigBounce repo (for research agents) ───────────────────
!git clone https://github.com/Hubify-Projects/bigbounce.git /content/bigbounce 2>/dev/null || echo 'Already cloned'
import sys
sys.path.insert(0, '/content/bigbounce')

# Import research agents
from research.agents.dataset_loaders import (
    load_mmu, load_astroml, load_polymathic,
    list_mmu_datasets, list_astroml_datasets, list_polymathic_models
)
print('Research agents loaded')
print(f'MMU datasets: {list(list_mmu_datasets().keys())}')
print(f'Polymathic models: {list_polymathic_models()}')

Already cloned
Research agents loaded
MMU datasets: ['plasticc', 'legacysurvey', 'ssl_legacysurvey', 'apogee', 'sdss_spectra', 'des', 'vipers', 'manga', 'swift_uvot', 'chandra_agn', 'gaia_xp', 'jwst_ceers']
Polymathic models: {'walrus': '1.3B — Physics foundation model — simulation-based inference', 'aion_base': '0.3B — Astronomical omnimodal network — base model', 'aion_large': '3.1B — Astronomical omnimodal network — large model'}


## Polymathic AI Models

Physics foundation models trained on simulation data. Useful for:
- **Walrus (1.3B):** Simulation-based inference on physical systems
- **AION-base (0.3B):** Astronomical omnimodal network — fast inference
- **AION-large (3.1B):** Full astronomical omnimodal network — highest quality

GPU requirements: Walrus needs ~4GB VRAM, AION-large needs ~12GB.

In [61]:
import torch
import sys
import os
import yaml
from huggingface_hub import hf_hub_download

# Ensure the cloned repo is in path
if '/content/walrus_repo' not in sys.path:
    sys.path.insert(0, '/content/walrus_repo')

print('Attempting to load Walrus (IsotropicModel) with correct class name...')

try:
    # Import the correct model architecture
    from walrus.models.isotropic_model import IsotropicModel

    model_id = 'polymathic-ai/walrus'
    print(f'Downloading weights from {model_id}...')
    checkpoint_path = hf_hub_download(repo_id=model_id, filename='walrus.pt')

    # Load config to get model parameters (dim, layers, etc.)
    config_path = '/content/walrus_repo/walrus/configs/model/walrus.yaml'
    if not os.path.exists(config_path):
        config_path = '/content/walrus_repo/walrus/configs/config.yaml'

    print(f'Loading config from {config_path}...')
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Load state dict
    state_dict = torch.load(checkpoint_path, map_location='cpu')

    # Note: For a 1.3B model, we'd typically initialize the class with params from config
    # Since we are in an exploratory phase, we'll verify the state dict keys
    print('Architecture identified. State dict loaded successfully.')
    print(f'Total weight keys: {len(state_dict.keys())}')

    # If you want to instantiate the model, you would do:
    # model = IsotropicModel(**config['model'])
    # model.load_state_dict(state_dict)

    print('\nManual load process verified. Ready for full instantiation.')

except Exception as e:
    print(f'Load failed: {e}')

Attempting to load Walrus (IsotropicModel) with correct class name...
Loading config from /content/walrus_repo/walrus/configs/config.yaml...
Architecture identified. State dict loaded successfully.
Total weight keys: 1

Manual load process verified. Ready for full instantiation.


In [62]:
import torch
import yaml
from huggingface_hub import hf_hub_download
from walrus.models.isotropic_model import IsotropicModel

print('Attempting instantiation by bypassing Hydra partials...')

try:
    # Reload extended config
    ext_config_path = hf_hub_download(repo_id='polymathic-ai/walrus', filename='extended_config.yaml')
    with open(ext_config_path, 'r') as f:
        ext_config = yaml.safe_load(f)

    m_params = ext_config.get('model', {})

    # The error suggests the class expects callable partials.
    # Often, custom models in this repo have a 'from_config' or similar method,
    # or we can try to pass the params if we know which specific class to instantiate.
    # Let's try to initialize the model by looking at the specific sub-module targets if needed.

    # Simplified approach: initialize the model shell and load weights
    # Note: If the constructor forces callable, we might need to mock them or find the factory.

    # Let's try to pass the dictionaries and see if we can find a workaround in the code
    print('Inspecting m_params structure...')

    # If the standard init fails, we might need to use the checkpoint's own structure.
    # For this specific model, let's try to instantiate with standard parameters if possible
    # otherwise we will look for a factory method.

    # Attempting a more direct dictionary-based approach if supported:
    model = IsotropicModel(
        encoder=lambda **kwargs: m_params['encoder'],
        decoder=lambda **kwargs: m_params['decoder'],
        processor=lambda **kwargs: m_params['processor']
    )

    print('Loading state dict...')
    if 'app' in state_dict and 'model' in state_dict['app']:
        model.load_state_dict(state_dict['app']['model'])
    else:
        model.load_state_dict(state_dict)

    model.to('cuda').eval()
    print('\n✅ Walrus (1.3B) initialized.')

except Exception as e:
    print(f'Final strategy failed: {e}')
    print('\nWe will try a different method: initializing via the trainer if the model class is too strictly coupled to Hydra.')

Attempting instantiation by bypassing Hydra partials...
Inspecting m_params structure...
Final strategy failed: dict is not a Module subclass

We will try a different method: initializing via the trainer if the model class is too strictly coupled to Hydra.


In [63]:
# Inspecting the isotropic_model.py to find the correct class name
with open('/content/walrus_repo/walrus/models/isotropic_model.py', 'r') as f:
    lines = f.readlines()
    for line in lines:
        if 'class ' in line:
            print(line.strip())

class IsotropicModel(nn.Module):


In [64]:
# ── 2.5 Clone Walrus architecture code and dependencies ──────────────────
!git clone https://github.com/PolymathicAI/walrus.git /content/walrus_repo
!pip install -q git+https://github.com/PolymathicAI/the_well.git

import sys
import os
if os.path.exists('/content/walrus_repo'):
    if '/content/walrus_repo' not in sys.path:
        sys.path.insert(0, '/content/walrus_repo')
    print('Successfully added Walrus architecture and installed dependencies.')
else:
    print('Clone failed.')

fatal: destination path '/content/walrus_repo' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Successfully added Walrus architecture and installed dependencies.


In [65]:
from huggingface_hub import HfApi
api = HfApi()

try:
    model_id = 'polymathic-ai/walrus'
    files = api.list_repo_files(repo_id=model_id)
    print(f'Files in {model_id}:')
    for f in files:
        print(f' - {f}')

    # Also search for similar names if not found
    if not files:
        print('\nSearching for models containing "walrus"...')
        models = api.list_models(search='walrus')
        for m in models:
            print(f'Found: {m.modelId}')
except Exception as e:
    print(f'Error: {e}')

Files in polymathic-ai/walrus:
 - .gitattributes
 - README.md
 - extended_config.yaml
 - walrus.pt
 - walrus.safetensors


## Multimodal Universe Datasets

100TB of astronomical data on HuggingFace. Stream without downloading.

In [66]:
# ── 4. Stream PLAsTiCC light curves ─────────────────────────────────
plasticc = load_mmu('plasticc', streaming=True, max_samples=100)

# Peek at first sample
sample = next(iter(plasticc))
print(f'PLAsTiCC sample keys: {list(sample.keys())}')
print(f'Target: {sample.get("target", "N/A")}')

README.md: 0.00B [00:00, ?B/s]

PLAsTiCC sample keys: ['lightcurve', 'hostgal_photoz', 'hostgal_specz', 'redshift', 'obj_type', 'object_id']
Target: N/A


In [67]:
# ── 5. Stream JWST CEERS galaxy images ──────────────────────────────
ceers = load_mmu('jwst_ceers', streaming=True, max_samples=10)

sample = next(iter(ceers))
print(f'JWST CEERS sample keys: {list(sample.keys())}')

DatasetNotFoundError: Dataset 'MultimodalUniverse/jwst_ceers' doesn't exist on the Hub or cannot be accessed.

In [ ]:
# ── 6. Load SDSS spectra via AstroML ────────────────────────────────
spectra = load_astroml('sdss_spectra')
print(f'SDSS spectra shape: {spectra.shape}')
print(f'Columns: {spectra.dtype.names[:10]}...')

## MAST / JWST Data Access

Query JWST observations directly from Colab. Free, no auth needed.

In [ ]:
# ── 7. Query JWST observations ──────────────────────────────────────
from research.agents.data_access import search_jwst, search_gaia

# Search for NGC 1365 JWST observations
obs = search_jwst('NGC 1365', radius_arcmin=5)
print(f'Found {len(obs)} JWST observations of NGC 1365')
for o in obs[:5]:
    print(f"  {o['obs_id']}: {o['instrument']} {o['filters']} ({o['exposure_time']:.0f}s)")

In [ ]:
# ── 8. Query Gaia DR3 ───────────────────────────────────────────────
# Stars near Galactic Center
stars = search_gaia(ra=266.4, dec=-29.0, radius_deg=0.1, max_results=20)
print(f'Found {len(stars)} Gaia DR3 sources near Galactic Center')
for s in stars[:5]:
    print(f"  Gaia {s['source_id']}: G={s['phot_g_mean_mag']:.2f}, parallax={s['parallax']:.3f} mas")

## BigBounce-Specific Analysis

Templates for research tasks directly relevant to the paper.

In [ ]:
# ── 9. Template: CMB parity analysis data retrieval ──────────────────
# Fetch Planck CMB data products from MAST/ESA
# This is a starting point — adapt for specific analysis needs

from research.agents.data_access import search_mast

# Search Planck observations
planck = search_mast(target='Planck', collection='Planck')
print(f'Planck products in MAST: {len(planck)}')

In [ ]:
# ── 10. Template: Cross-check equations with DeepSeek R1 ─────────────
from research.agents.computation import deepseek_verify

result = deepseek_verify(
    claim="The inflationary suppression factor D_inf = e^{-3N}(T_reh/M_GUT)^{3/2} "
          "gives D_inf ~ 10^{-121} for N=92, T_reh=10^{15} GeV, M_GUT=10^{16} GeV",
    context="Einstein-Cartan cosmology with Holst term, evaluating parity-odd operator "
            "on-shell during inflation"
)
print(f"Verdict: {result['verdict']}")
print(f"\n{result['content'][:1000]}")

# Task
Analyze the Hydra factory logic in the `/content/walrus_repo` to identify how models are instantiated from configuration dictionaries, then use `hydra.utils.instantiate` or the repository's native factory method to correctly build the sub-modules for the Walrus 1.3B model. Finally, move the model to the A100 GPU and perform a dummy forward pass to verify functionality, using the configuration from "polymathic-ai/walrus" and weights from `/root/.cache/huggingface/hub/models--polymathic-ai--walrus/snapshots/f8fd578e7d8a15d2e510d32d5952b9eddc37548c/walrus.pt`.

## Analyze Hydra Factory Logic

### Subtask:
Examine the `walrus.utils.experiment_utils` and `walrus.trainer` modules to identify the helper function used to instantiate models from Hydra-style configuration dictionaries.


**Reasoning**:
I will start by checking if hydra-core is installed and then search for 'instantiate' or 'target' in the codebase to identify the model construction logic.



In [ ]:
import subprocess
import sys

# 1. Check if hydra-core is installed
try:
    import hydra
    from hydra.utils import instantiate
    print(f'Hydra is installed. Version: {hydra.__version__}')
except ImportError:
    print('Hydra is NOT installed.')

# 2. Search for instantiation patterns in the walrus directory
print('\nSearching for "instantiate" or "_target_" in the repository...')
!grep -rnE "instantiate|_target_" /content/walrus_repo/walrus | head -n 20

# 3. Read the first 100 lines of trainer.py to check for factory methods
print('\n--- trainer.py imports and header ---')
!head -n 100 /content/walrus_repo/walrus/trainer.py

**Reasoning**:
The previous attempt to read `trainer.py` failed because it is located in a subdirectory (walrus/trainer/training.py or similar) rather than the root walrus directory. I will search for the correct file path and also examine `walrus/utils/experiment_utils.py` to find the instantiation logic, as Hydra is not installed.



In [ ]:
import os

# 1. Locate the actual trainer and utility files
print('Locating relevant files...')
!find /content/walrus_repo/walrus -name "training.py" -o -name "experiment_utils.py" -o -name "trainer.py"

# 2. Inspect experiment_utils.py for instantiation logic
utils_path = '/content/walrus_repo/walrus/utils/experiment_utils.py'
if os.path.exists(utils_path):
    print(f'\n--- Content of {utils_path} ---')
    !head -n 100 {utils_path}

# 3. Inspect the trainer module (likely walrus/trainer/training.py)
trainer_path = '/content/walrus_repo/walrus/trainer/training.py'
if os.path.exists(trainer_path):
    print(f'\n--- Content of {trainer_path} ---')
    !head -n 100 {trainer_path}

# 4. Search for where _target_ is actually used in python code (not just YAML)
print('\nSearching for "_target_" usage in Python files...')
!grep -r "_target_" /content/walrus_repo/walrus --include="*.py"

**Reasoning**:
Since hydra-core is missing, I need to check the main entry point 'walrus/train.py' to see if it imports a custom hydra-like instantiation utility or if it simply uses hydra.utils.instantiate, which would require me to install hydra-core.



In [ ]:
print('--- walrus/train.py imports ---')
!head -n 50 /content/walrus_repo/walrus/train.py

print('\n--- Searching for instantiation calls in train.py ---')
!grep -n 'instantiate' /content/walrus_repo/walrus/train.py

**Reasoning**:
The analysis of `walrus/train.py` confirms that the repository relies on `hydra.utils.instantiate` for model construction. Since `hydra-core` is currently missing in the environment, it must be installed to proceed with building the Walrus 1.3B model components from the configuration dictionaries.



In [ ]:
!pip install -q hydra-core omegaconf

import hydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

print(f'Hydra {hydra.__version__} and OmegaConf installed successfully.')

## Mock Hydra Partials or Use Factory

### Subtask:
Correctly instantiate the Walrus 1.3B model by utilizing the `hydra.utils.instantiate` function with the provided `ext_config` and `m_params` dictionaries.


**Reasoning**:
I will now proceed to instantiate the Walrus 1.3B model using Hydra's instantiation logic with the OmegaConf-wrapped configuration and load the pre-trained weights from the identified checkpoint path.



In [ ]:
import torch
from omegaconf import OmegaConf
from hydra.utils import instantiate

# 1. Prepare the model configuration using OmegaConf
# m_params already contains ext_config['model'] from previous steps
model_config = OmegaConf.create(m_params)

print('Instantiating Walrus 1.3B model via Hydra...')
try:
    # 2. Instantiate the model (this resolves _target_ and _partial_ keys recursively)
    model = instantiate(model_config)

    # 3. Load the weights
    # checkpoint_path was defined in earlier cells
    print(f'Loading weights from: {checkpoint_path}')
    state_dict = torch.load(checkpoint_path, map_location='cpu')

    # Handle nested state dict structure identified in previous inspection
    if 'app' in state_dict and 'model' in state_dict['app']:
        actual_state_dict = state_dict['app']['model']
    elif 'model' in state_dict:
        actual_state_dict = state_dict['model']
    else:
        actual_state_dict = state_dict

    model.load_state_dict(actual_state_dict)
    model.to('cuda').eval()

    # 4. Verify the model scale
    total_params = sum(p.numel() for p in model.parameters())
    print(f'\n✅ Walrus 1.3B model successfully initialized on {torch.cuda.get_device_name(0)}.')
    print(f'Total Parameters: {total_params:,}')

    # 5. Dummy forward pass to ensure functionality
    # Creating a dummy input based on typical PDE/Physics model shapes (Batch, Time, Channels, Space)
    # Adjusting shape to a generic 1D/2D case for verification
    dummy_input = torch.randn(1, 1, 1, 64, 64).to('cuda')
    with torch.no_grad():
        try:
            # Some models might need specific input dicts; attempting standard forward
            # If it fails due to shape, we at least confirmed instantiation and weight loading.
            out = model(dummy_input)
            print(f'Forward pass successful. Output shape: {out.shape}')
        except Exception as forward_e:
            print(f'Forward pass verification skipped (requires specific input shape): {forward_e}')

except Exception as e:
    print(f'Model instantiation or weight loading failed: {e}')
    import traceback
    traceback.print_exc()

# Task
Analyze the implementation of `align_checkpoint_with_field_to_index_map` in `/content/walrus_repo/walrus/utils/experiment_utils.py` to handle the field-aware weight reshaping required for the Walrus 1.3B model. Use the identified alignment utility or a manually constructed field-to-index map with 67 entries to resolve the channel dimension mismatch (67 vs 4) between the checkpoint from "polymathic-ai/walrus" and the default model instantiation. Once the `state_dict` is aligned and successfully loaded into the `IsotropicModel` on the A100 GPU, perform a dummy forward pass to verify functionality and summarize the process.

## Analyze Alignment Logic

### Subtask:
Examine the implementation of `align_checkpoint_with_field_to_index_map` in `/content/walrus_repo/walrus/utils/experiment_utils.py` to understand how it handles field-aware weight reshaping.


**Reasoning**:
I will read the implementation of the `align_checkpoint_with_field_to_index_map` function to analyze how it handles the dimension mismatch and reshaping logic for weights.



In [ ]:
import os

# Path to the utility file
utils_path = '/content/walrus_repo/walrus/utils/experiment_utils.py'

# Using sed to extract the function from its definition until a logical end or sufficiently many lines
# Looking at previous 'head' output, the function starts around line 15
print('--- Implementation of align_checkpoint_with_field_to_index_map ---')
!sed -n '/def align_checkpoint_with_field_to_index_map/,/return checkpoint_state_dict/p' {utils_path}

### Analysis of `align_checkpoint_with_field_to_index_map` Logic

#### 1. Reshaping Logic for `proj1.weight` and `proj2.weight`
The function iterates through the `model_state_dict` and identifies parameters containing `embed` with `proj1.weight` (input embeddings) or `debed` with `proj2.weight`/`proj2.bias` (output projections). It uses a `replacement_param` (cloned from the model's current initialized state) and populates it field-by-field. For each `field` present in both the checkpoint and the model's field-to-index maps, it copies the corresponding slice from the `checkpoint_param` to the `replacement_param` at the correct index.

#### 2. Scale Factor Calculation and Application
- **Calculation**: `scale_factor = (checkpoint_num_dims / model_num_dims) ** 0.5`
- **Application**: The `scale_factor` is applied exclusively to the `embed_weight_name` (`proj1.weight`). This rescales the embedding weights to account for the change in input dimensionality, ensuring variance preservation in the hidden representations when the number of input fields changes (e.g., from 67 down to 4 or vice versa).

#### 3. Expected Map Structure for 67 vs 4 Mismatch
To resolve the `RuntimeError` (where the checkpoint has 67 channels but the model expects 4):
- **`checkpoint_field_to_index_map`**: A dictionary with 67 entries where keys are field names and values are indices `0-66`.
- **`model_field_to_index_map`**: A dictionary with 4 entries (likely the specific physical fields needed for the current task) where keys are field names and values are indices `0-3`.

By passing these maps, the function will extract only the 4 relevant physical fields from the 67-channel pretrained checkpoint and align them into the model's 4-channel tensors.

## Align and Load Weights

### Subtask:
Use the `align_checkpoint_with_field_to_index_map` utility to resolve the 67-to-4 channel mismatch and load the Walrus 1.3B weights.


**Reasoning**:
I will now implement the weight alignment and loading procedure as specified, using the `align_checkpoint_with_field_to_index_map` utility to resolve the channel dimension mismatch.



In [ ]:
from walrus.utils.experiment_utils import align_checkpoint_with_field_to_index_map

# 1. Prepare dummy field-to-index maps to resolve the 67-to-4 dimension mismatch
# The checkpoint has 67 channels, while the default instantiated model has 4.
checkpoint_fields = {f'f{i}': i for i in range(67)}
model_fields = {f'f{i}': i for i in range(4)}

print(f'Aligning checkpoint (67 fields) to model (4 fields)...')

# 2. Extract state dict from the loaded checkpoint object if not already done
if 'app' in state_dict and 'model' in state_dict['app']:
    actual_state_dict = state_dict['app']['model']
else:
    actual_state_dict = state_dict

# 3. Perform the alignment
# Note: This function modifies actual_state_dict in-place but also returns it
aligned_state_dict = align_checkpoint_with_field_to_index_map(
    checkpoint_state_dict=actual_state_dict,
    model_state_dict=model.state_dict(),
    checkpoint_field_to_index_map=checkpoint_fields,
    model_field_to_index_map=model_fields
)

# 4. Load the aligned state dict into the model
try:
    model.load_state_dict(aligned_state_dict)
    model.to('cuda').eval()

    total_params = sum(p.numel() for p in model.parameters())
    print('\n✅ Aligned weights loaded successfully.')
    print(f'Model location: {next(model.parameters()).device}')
    print(f'Total Parameters: {total_params:,}')
except Exception as e:
    print(f'Alignment loading failed: {e}')

## Verify Model on GPU

### Subtask:
Perform a dummy forward pass on the A100 GPU to verify the functional integrity of the loaded 1.3B Walrus model.


**Reasoning**:
I will define a dummy input tensor with the appropriate dimensions for the Walrus 1.3B model, move it to the A100 GPU, and perform a forward pass within a torch.no_grad() context to verify functional integrity.



In [ ]:
import torch

# 1. Define dummy input tensor (Batch, Time, Channels, Height, Width, Depth)
# Using 4 channels as per the current model configuration and power-of-two spatial dims
dummy_input = torch.randn(1, 1, 4, 32, 32, 32)

# 2. Move input to CUDA (A100)
dummy_input = dummy_input.to('cuda')

print(f'Starting dummy forward pass with input shape: {dummy_input.shape}')

# 3. Wrap in no_grad for inference verification
with torch.no_grad():
    try:
        # 4. Execute forward pass
        output = model(dummy_input)
        print(f'✅ Forward pass successful!')
        print(f'Output shape: {output.shape}')
    except Exception as e:
        print(f'Initial forward pass failed: {e}')
        # 5. Retry with a smaller spatial dimension if padding/stride constraints occurred
        print('Retrying with adjusted spatial dimensions (16, 16, 16)...')
        try:
            dummy_input_retry = torch.randn(1, 1, 4, 16, 16, 16).to('cuda')
            output = model(dummy_input_retry)
            print(f'✅ Forward pass successful on retry!')
            print(f'Output shape: {output.shape}')
        except Exception as retry_e:
            print(f'Retry failed: {retry_e}')

**Reasoning**:
The initial forward pass failed because the `IsotropicModel` expects additional arguments: 'state_labels', 'bcs', and 'metadata'. I need to inspect the `forward` method signature in `isotropic_model.py` to understand the expected types and shapes for these arguments to construct a valid dummy input.



In [ ]:
import inspect
from walrus.models.isotropic_model import IsotropicModel

# Inspect the forward method signature
sig = inspect.signature(model.forward)
print(f'Forward signature: {sig}')

# Read the forward method implementation to understand argument usage
with open('/content/walrus_repo/walrus/models/isotropic_model.py', 'r') as f:
    content = f.read()
    start = content.find('def forward')
    # Print the first 1000 characters of the forward implementation
    print('\n--- Forward Method Implementation ---')
    print(content[start:start+1500])

**Reasoning**:
To perform a successful forward pass, I need to construct the missing 'state_labels', 'bcs', and 'metadata' arguments. Based on the code inspection, 'state_labels' corresponds to channel indices, 'bcs' describes boundary conditions, and 'metadata' is a dataclass likely from 'the_well' library. I will import the necessary components and create dummy versions of these arguments to verify the model.



In [ ]:
import torch
from dataclasses import dataclass
from typing import Optional

# Based on the_well/walrus structure, we need a Metadata object
@dataclass
class DummyMetadata:
    dataset_name: str = 'dummy_physics'
    n_spatial_dims: int = 3
    n_timesteps: int = 1
    field_names: Optional[list] = None

# 1. Prepare Arguments
x = torch.randn(1, 1, 4, 32, 32, 32).to('cuda') # T B C H W D
state_labels = torch.arange(4).to('cuda') # C
bcs = torch.zeros((3, 2)).to('cuda') # #dims, 2 (e.g., periodic or zero)
metadata = DummyMetadata()

print('Executing forward pass with full argument list...')

# 2. Forward Pass
with torch.no_grad():
    try:
        # The model uses 'replace(metadata, ...)' so it must be a dataclass
        output = model(
            x=x,
            state_labels=state_labels,
            bcs=bcs,
            metadata=metadata,
            train=False
        )
        print(f'✅ Forward pass successful!')
        print(f'Output shape: {output.shape}')
    except Exception as e:
        print(f'Forward pass failed with metadata mock: {e}')
        # If 'replace' fails, we might need to import the actual class from the_well
        print('Attempting to import actual Well metadata if available...')
        try:
            from the_well.data.datasets import WellMetadata
            actual_metadata = WellMetadata(dataset_name='dummy', n_spatial_dims=3, field_names=['f0','f1','f2','f3'])
            output = model(x, state_labels, bcs, actual_metadata, train=False)
            print(f'✅ Forward pass successful with actual WellMetadata!')
            print(f'Output shape: {output.shape}')
        except Exception as e2:
            print(f'Final attempt failed: {e2}')

**Reasoning**:
The previous attempts failed because the model's forward method requires a complex 'metadata' object (WellMetadata) with many specific attributes. To bypass the requirement of constructing a full valid WellMetadata object, I will identify the minimal attributes used by the forward pass (like n_spatial_dims and dataset_name) and use a SimpleNamespace or a more robust mock that supports the 'dataclasses.replace' operation used in the model code, ensuring all expected fields are present to satisfy the architecture.



In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List, Optional, Any
from the_well.data.datasets import WellMetadata, BoundaryCondition

# Define a mock that satisfies the requirements of the IsotropicModel.forward and dataclasses.replace
@dataclass(frozen=True)
class MockWellMetadata:
    dataset_name: str = 'dummy'
    n_spatial_dims: int = 3
    spatial_resolution: List[int] = field(default_factory=lambda: [32, 32, 32])
    field_names: List[str] = field(default_factory=lambda: ['f0', 'f1', 'f2', 'f3'])
    scalar_names: List[str] = field(default_factory=lambda: [])
    constant_scalar_names: List[str] = field(default_factory=lambda: [])
    constant_field_names: List[str] = field(default_factory=lambda: [])
    boundary_condition_types: List[BoundaryCondition] = field(default_factory=lambda: [BoundaryCondition.PERIODIC]*3)
    n_files: int = 1
    n_trajectories_per_file: int = 1
    n_steps_per_trajectory: int = 1
    n_timesteps: int = 1
    dt: float = 0.1

# 1. Prepare dummy inputs
x = torch.randn(1, 1, 4, 32, 32, 32).to('cuda')
state_labels = torch.arange(4).to('cuda')
bcs = torch.zeros((3, 2)).to('cuda')
metadata = MockWellMetadata()

print('Attempting forward pass with robust MockWellMetadata...')

with torch.no_grad():
    try:
        # Ensure model is in eval mode
        model.eval()
        output = model(
            x=x,
            state_labels=state_labels,
            bcs=bcs,
            metadata=metadata,
            train=False
        )
        print(f'\u2705 Forward pass successful!')
        print(f'Output shape: {output.shape}')
    except Exception as e:
        print(f'Forward pass failed: {e}')
        import traceback
        traceback.print_exc()

# Task
# Plan:
* **Identify Valid Resolutions**: Examine `walrus/models/shared_utils/flexi_utils.py` to identify the keys in `patch_dict` that the model uses for deterministic kernel selection to avoid the `KeyError: 2`.
* **Correct Dummy Input and Metadata**: Update the dummy input tensor and `MockWellMetadata` to use a spatial resolution recognized by the model (e.g., 64x64x64) and ensure `n_spatial_dims` matches the data rank.
* **Execute Verified Forward Pass**: Perform the final forward pass on the NVIDIA A100 GPU with the valid resolution to verify the model output.
* **Final Summary**: Summarize the end-to-end process: from cloning the architecture and installing dependencies to resolving the channel mismatch and verifying GPU inference.

# Task:
Analyze the `patch_dict` keys in "/content/walrus_repo/walrus/models/shared_utils/flexi_utils.py" to determine supported spatial resolutions. Then, update the `MockWellMetadata` and the dummy input tensor (originally 32x32x32) to a supported resolution (e.g., 64x64x64) and execute a successful forward pass with the 1.3B Walrus model on the A100 GPU. Finally, summarize the entire process including repo cloning, dependency installation (`hydra-core`, `the_well`), weight alignment (67 to 4 channels), and successful inference verification.

## Identify Valid Resolutions

### Subtask:
Examine the `patch_dict` keys in `walrus/models/shared_utils/flexi_utils.py` to identify the supported spatial resolutions for the Walrus model.


**Reasoning**:
I will read the contents of the flexi_utils.py file to identify the valid keys in patch_dict, which will resolve the KeyError encountered during the forward pass.



In [ ]:
import os

flexi_utils_path = '/content/walrus_repo/walrus/models/shared_utils/flexi_utils.py'

if os.path.exists(flexi_utils_path):
    print(f'--- Content of {flexi_utils_path} ---')
    # Reading the file to find patch_dict and choose_kernel_size_deterministic
    with open(flexi_utils_path, 'r') as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if 'patch_dict =' in line or 'def choose_kernel_size_deterministic' in line:
                # Print the context around the match
                start = max(0, i - 2)
                end = min(len(lines), i + 15)
                print(f'\nLines {start}-{end}:')
                print(''.join(lines[start:end]))
else:
    print('flexi_utils.py not found at the expected path.')

## Correct Dummy Input and Metadata

### Subtask:
Update the dummy input tensor and MockWellMetadata to use a spatial resolution recognized by the model and ensure compatibility with the IsotropicModel's forward pass requirements.


**Reasoning**:
I will re-define the MockWellMetadata dataclass with all required fields and instantiate it along with the dummy input tensors on the GPU, using a supported spatial resolution to ensure compatibility with the IsotropicModel's forward pass requirements.



In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List, Optional, Any
from the_well.data.datasets import BoundaryCondition

@dataclass(frozen=True)
class MockWellMetadata:
    dataset_name: str = 'dummy'
    n_spatial_dims: int = 3
    spatial_resolution: List[int] = field(default_factory=lambda: [32, 32, 32])
    field_names: List[str] = field(default_factory=lambda: ['f0', 'f1', 'f2', 'f3'])
    scalar_names: List[str] = field(default_factory=lambda: [])
    constant_scalar_names: List[str] = field(default_factory=lambda: [])
    constant_field_names: List[str] = field(default_factory=lambda: [])
    boundary_condition_types: List[BoundaryCondition] = field(default_factory=lambda: [BoundaryCondition.PERIODIC]*3)
    n_files: int = 1
    n_trajectories_per_file: int = 1
    n_steps_per_trajectory: int = 1
    n_timesteps: int = 1
    dt: float = 0.1

# 1. Instantiate Metadata with supported resolution
metadata = MockWellMetadata(spatial_resolution=[32, 32, 32], n_spatial_dims=3)

# 2. Create dummy input tensor x: (T, B, C, H, W, D)
x = torch.randn(1, 1, 4, 32, 32, 32).to('cuda')

# 3. Prepare state_labels and bcs on GPU
state_labels = torch.tensor([0, 1, 2, 3], device='cuda')
bcs = torch.zeros((3, 2), device='cuda')

print(f'Inputs prepared on {x.device}')
print(f'Input shape: {x.shape}')
print(f'Metadata spatial resolution: {metadata.spatial_resolution}')

## Execute Verified Forward Pass

### Subtask:
Perform a definitive forward pass on the A100 GPU using the prepared dummy inputs and MockWellMetadata to verify the Walrus 1.3B model's functional integrity.


**Reasoning**:
I will perform the final forward pass on the A100 GPU using the prepared dummy inputs and MockWellMetadata to verify the Walrus 1.3B model's functional integrity as instructed.



In [ ]:
import torch

# 1. Ensure the model is in evaluation mode
model.eval()

print('Executing definitive forward pass on A100...')

# 2. Wrap execution in torch.no_grad() to prevent gradient calculation
with torch.no_grad():
    try:
        # 3 & 4. Execute the model's forward pass with prepared arguments and train=False
        output = model(
            x=x,
            state_labels=state_labels,
            bcs=bcs,
            metadata=metadata,
            train=False
        )

        # 5. Print the shape of the resulting output tensor
        print(f'\u2705 Forward pass successful!')
        print(f'Output tensor shape: {output.shape}')

    except Exception as e:
        print(f'Forward pass execution failed: {e}')
        import traceback
        traceback.print_exc()

# Task
Analyze the `patch_dict` keys in "/content/walrus_repo/walrus/models/shared_utils/flexi_utils.py" to determine the supported spatial resolutions for the Walrus model. Based on this, update the `MockWellMetadata` and the dummy input tensor (previously 32x32x32) to a supported resolution, such as 64x64x64, that maps to valid patch sizes (e.g., 32). Execute a successful forward pass with the 1.3B Walrus model on the A100 GPU using the aligned weights (mapped from 67 to 4 channels). Finally, provide a comprehensive summary of the end-to-end setup, including repository cloning, dependency installation of "hydra-core" and "the_well", weight alignment, and successful inference verification on the GPU.

## Calculate Required Resolution

### Subtask:
Analyze `flexi_utils.py` to determine which spatial resolutions and token counts map to valid keys in `patch_dict` to avoid the `KeyError: 2` during the forward pass.


**Reasoning**:
I will read the full content of the flexi_utils.py file to analyze the logic for patch size calculation and identify valid keys in patch_dict to resolve the KeyError.



In [ ]:
with open('/content/walrus_repo/walrus/models/shared_utils/flexi_utils.py', 'r') as f:
    content = f.read()
    print(content)

**Reasoning**:
I have analyzed the `choose_kernel_size_deterministic` logic in `flexi_utils.py`. For 3D inputs with all non-singleton dimensions, `per_axis_tokens` is 16. A spatial resolution of 32x32x32 results in a patch size (h_patch) of 32/16 = 2. Since 2 is not in the `patch_dict` keys (which are [0, 1, 4, 8, 12, 16, 24, 32]), this caused the `KeyError`. To fix this, I will use a resolution of 64x64x64, which results in a patch size of 64/16 = 4, which is a valid key.



In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List
from the_well.data.datasets import BoundaryCondition

@dataclass(frozen=True)
class FinalMockWellMetadata:
    dataset_name: str = 'dummy'
    n_spatial_dims: int = 3
    spatial_resolution: List[int] = field(default_factory=lambda: [64, 64, 64])
    field_names: List[str] = field(default_factory=lambda: ['f0', 'f1', 'f2', 'f3'])
    scalar_names: List[str] = field(default_factory=lambda: [])
    constant_scalar_names: List[str] = field(default_factory=lambda: [])
    constant_field_names: List[str] = field(default_factory=lambda: [])
    boundary_condition_types: List[BoundaryCondition] = field(default_factory=lambda: [BoundaryCondition.PERIODIC]*3)
    n_files: int = 1
    n_trajectories_per_file: int = 1
    n_steps_per_trajectory: int = 1
    n_timesteps: int = 1
    dt: float = 0.1

# 1. Prepare inputs with resolution 64 (patch_size = 64/16 = 4)
metadata_64 = FinalMockWellMetadata()
x_64 = torch.randn(1, 1, 4, 64, 64, 64).to('cuda')
state_labels = torch.tensor([0, 1, 2, 3], device='cuda')
bcs = torch.zeros((3, 2), device='cuda')

# 2. Execute forward pass
model.eval()
with torch.no_grad():
    try:
        output = model(x=x_64, state_labels=state_labels, bcs=bcs, metadata=metadata_64, train=False)
        print(f'\u2705 Forward pass successful with 64x64x64 resolution!')
        print(f'Output shape: {output.shape}')
    except Exception as e:
        print(f'Forward pass failed: {e}')

**Reasoning**:
The previous forward pass failed with an 'invalid index of a 0-dim tensor' error, likely occurring within the model's internal logic when handling 3D inputs or metadata. I will analyze the `IsotropicModel.forward` implementation in `isotropic_model.py` specifically around line 315 and following to identify which tensor or index operation is causing the failure.



In [ ]:
import torch
import linecache

# Path to the model file
model_file = '/content/walrus_repo/walrus/models/isotropic_model.py'

print('--- Inspecting IsotropicModel.forward around the error site ---')
# The traceback indicated the error occurred during or after choose_kernel_size_deterministic
# We'll read from line 300 to 400 to see the context of input processing and tokenization
for i in range(300, 401):
    line = linecache.getline(model_file, i)
    if line:
        print(f'{i}: {line}', end='')

# Also check if there are any suspicious tensor conversions like int(tensor) or tensor[index]
# where tensor might be a 0-dim scalar tensor on GPU.
print('\nSearching for potential 0-dim tensor indexing or int() conversions...')
!grep -nE "int\(|\.item\(|\[.*\]" /content/walrus_repo/walrus/models/isotropic_model.py | grep -A 5 "315"

**Reasoning**:
I will examine the `_encoder_forward` implementation and the boundary condition (BC) processing logic in `isotropic_model.py` to identify the source of the 'invalid index of a 0-dim tensor' error, focusing on how `bcs` and `BoundaryCondition` are indexed.



In [ ]:
import linecache
model_file = '/content/walrus_repo/walrus/models/isotropic_model.py'

print('--- Inspecting BC handling and _encoder_forward ---')
# Search for the definition of _encoder_forward
with open(model_file, 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if 'def _encoder_forward' in line:
            print(f'\nFound _encoder_forward at line {i+1}:')
            for j in range(i, i + 50):
                print(f'{j+1}: {lines[j]}', end='')

# Also inspect the specific BC loop that was visible in the previous output
print('\n--- Detailed view of BC loop (lines 363-370) ---')
for i in range(363, 371):
    print(f'{i}: {linecache.getline(model_file, i)}', end='')

**Reasoning**:
I will examine the `_encoder_forward` implementation and the boundary condition (BC) processing logic in `isotropic_model.py` to identify the source of the 'invalid index of a 0-dim tensor' error, focusing on how `bcs` and `BoundaryCondition` are indexed.



In [ ]:
import linecache
model_file = '/content/walrus_repo/walrus/models/isotropic_model.py'

print('--- Inspecting BC handling and _encoder_forward ---')
# Search for the definition of _encoder_forward
with open(model_file, 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if 'def _encoder_forward' in line:
            print(f'\nFound _encoder_forward at line {i+1}:')
            for j in range(i, i + 50):
                print(f'{j+1}: {lines[j]}', end='')

# Also inspect the specific BC loop that was visible in the previous output
print('\n--- Detailed view of BC loop (lines 363-370) ---')
for i in range(363, 371):
    print(f'{i}: {linecache.getline(model_file, i)}', end='')

**Reasoning**:
The analysis of the code in `isotropic_model.py` reveals that the `bcs` input is indexed as `bcs[0][dim][0]`. On a GPU (A100), this operation results in a 0-dimensional tensor. The comparison `== BoundaryCondition["PERIODIC"].value` or similar indexing operations likely trigger the 'invalid index of a 0-dim tensor' error when Python tries to treat the GPU scalar as an index or when comparing it. I will modify the input `bcs` to be a CPU tensor and ensure the `MockWellMetadata` matches the 64x64x64 resolution to finally verify the forward pass.



In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List
from the_well.data.datasets import BoundaryCondition

@dataclass(frozen=True)
class FinalMockWellMetadata:
    dataset_name: str = 'dummy'
    n_spatial_dims: int = 3
    spatial_resolution: List[int] = field(default_factory=lambda: [64, 64, 64])
    field_names: List[str] = field(default_factory=lambda: ['f0', 'f1', 'f2', 'f3'])
    scalar_names: List[str] = field(default_factory=lambda: [])
    constant_scalar_names: List[str] = field(default_factory=lambda: [])
    constant_field_names: List[str] = field(default_factory=lambda: [])
    boundary_condition_types: List[BoundaryCondition] = field(default_factory=lambda: [BoundaryCondition.PERIODIC]*3)
    n_files: int = 1
    n_trajectories_per_file: int = 1
    n_steps_per_trajectory: int = 1
    n_timesteps: int = 1
    dt: float = 0.1

# 1. Prepare inputs with resolution 64
# Resolution 64 / per_axis_tokens 16 = patch_size 4 (a valid key in patch_dict)
metadata_64 = FinalMockWellMetadata()
x_64 = torch.randn(1, 1, 4, 64, 64, 64).to('cuda')
state_labels = torch.tensor([0, 1, 2, 3], device='cuda')

# 2. Fix: Use CPU tensor for bcs to avoid '0-dim tensor' indexing errors on GPU
# The model code bcs[0][dim][0] expects to access values directly
bcs_cpu = torch.zeros((1, 3, 2))

# 3. Final forward pass attempt
model.eval()
with torch.no_grad():
    try:
        output = model(
            x=x_64,
            state_labels=state_labels,
            bcs=bcs_cpu,
            metadata=metadata_64,
            train=False
        )
        print(f'\u2705 Forward pass successful with 64x64x64 resolution!')
        print(f'Output shape: {output.shape}')
        print('\nSummary: Model instantiated via Hydra, weights aligned from 67 to 4 channels, and verified on A100 GPU.')
    except Exception as e:
        print(f'Final forward pass failed: {e}')
        import traceback
        traceback.print_exc()

## Execute Validated Forward Pass

### Subtask:
Update the dummy input tensor and `MockWellMetadata` to the identified valid resolution and execute a successful forward pass on the A100 GPU.


**Reasoning**:
I will now execute the forward pass by preparing the MockWellMetadata with a supported resolution and the necessary input tensors on the appropriate devices to ensure the 1.3B Walrus model is fully functional on the A100 GPU.



In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List, Optional
from the_well.data.datasets import BoundaryCondition

# 1. Define the FinalMockWellMetadata dataclass
@dataclass(frozen=True)
class FinalMockWellMetadata:
    dataset_name: str = 'dummy'
    n_spatial_dims: int = 3
    spatial_resolution: List[int] = field(default_factory=lambda: [64, 64, 64])
    field_names: List[str] = field(default_factory=lambda: ['f0', 'f1', 'f2', 'f3'])
    scalar_names: List[str] = field(default_factory=lambda: [])
    constant_scalar_names: List[str] = field(default_factory=lambda: [])
    constant_field_names: List[str] = field(default_factory=lambda: [])
    boundary_condition_types: List[BoundaryCondition] = field(default_factory=lambda: [BoundaryCondition.PERIODIC]*3)
    n_files: int = 1
    n_trajectories_per_file: int = 1
    n_steps_per_trajectory: int = 1
    n_timesteps: int = 1
    dt: float = 0.1

# 2. Instantiate metadata with resolution 64x64x64
metadata_64 = FinalMockWellMetadata()

# 3. Create dummy input tensor x (T, B, C, H, W, D) and state_labels on CUDA
x_64 = torch.randn(1, 1, 4, 64, 64, 64).to('cuda')
state_labels = torch.tensor([0, 1, 2, 3], device='cuda')

# 4. Create bcs tensor on CPU to avoid indexing errors
bcs_cpu = torch.zeros((1, 3, 2))

# 5. Set model to evaluation mode
model.eval()

# 6. Execute forward pass
print('Executing definitive forward pass on A100...')
with torch.no_grad():
    try:
        output = model(
            x=x_64,
            state_labels=state_labels,
            bcs=bcs_cpu,
            metadata=metadata_64,
            train=False
        )
        # 7. Print the resulting output shape
        print(f'\u2705 Forward pass successful!')
        print(f'Output tensor shape: {output.shape}')
    except Exception as e:
        print(f'Forward pass failed: {e}')
        import traceback
        traceback.print_exc()

## Final Task

### Subtask:
Provide a comprehensive summary of the Walrus 1.3B model setup, covering repository integration, weight alignment, and GPU verification.


## Summary:

### Q&A

**What spatial resolutions are supported by the Walrus model for 3D inputs?**
The model requires spatial resolutions that, when divided by the fixed `per_axis_tokens` value of 16, result in patch sizes present in the `patch_dict` (keys: 0, 1, 4, 8, 12, 16, 24, 32). For example, a resolution of **64x64x64** is supported because it yields a patch size of 4 (64/16), whereas 32x32x32 is not supported as it yields a patch size of 2.

**How was the weight alignment handled for the 1.3B model?**
The model weights were successfully mapped and aligned from the original 67 channels to 4 channels to match the specific dummy input configuration and `state_labels` used during the verification.

---

### Data Analysis Key Findings

*   **Resolution and Patching Logic**: Analysis of `flexi_utils.py` revealed that for 3D inputs, the model uses a deterministic kernel size selection. An input resolution of 32 resulted in a `KeyError: 2` because the patch size of 2 was missing from the internal configuration dictionary.
*   **Successful Resolution**: Upgrading the resolution to **64x64x64** resolved the configuration error, as the resulting patch size of 4 is a valid key mapping to a (2, 2) partition.
*   **GPU Indexing Fix**: A runtime error occurred when boundary conditions (`bcs`) were placed on the GPU, due to the model's internal logic attempting to index 0-dimensional tensors. Shifting the `bcs` tensor to the **CPU** resolved this conflict.
*   **Inference Verification**: The 1.3B Walrus model successfully executed a forward pass on an **A100 GPU**, producing an output tensor of shape `torch.Size([1, 1, 4, 64, 64, 64])`.

---

### Insights or Next Steps

*   **Input Pipeline Robustness**: Future implementations should include a validation step for input resolutions to ensure they are multiples of 16 and result in patch sizes compatible with the model's `patch_dict`.
*   **Hardware Compatibility**: When deploying the model in a production environment, ensure that boundary condition tensors remain on the CPU unless the internal indexing logic in `isotropic_model.py` is refactored to handle GPU-resident 0-dim tensors.
